In [1]:
import ipywidgets as widgets
from IPython.display import display
import datetime, json, uuid

# ============================================================
# PRO173 – Mantenimiento Correctivo
# HMI oficial (BASE estilo PRO134 aprobado)
# ============================================================

EN_CURSO = "EN_CURSO"
BLOQUEADO = "BLOQUEADO"
DETENIDO_STOP = "DETENIDO_STOP"
FINALIZADO = "FINALIZADO"

def _now_iso():
    return datetime.datetime.now().isoformat(timespec="seconds")

MOTIVOS_BLOQUEO_PRO173 = [
    "PT no vigente / condiciones de PT no cumplen",
    "Condición de seguridad no controlada (energías, accesos, EPP, etc.)",
    "Recursos no disponibles (personal / herramientas / repuestos / servicios)",
    "OT / Aviso inconsistente (prioridad, alcance, UT/Proceso, operaciones)",
    "Sistema SAP indisponible / sin acceso (IW31 / IW32 / IW41 / IW48)",
    "No se puede ejecutar según planificación (ventana, coordinación, permisos)",
    "Requiere autorización / aprobación no obtenida (servicios externos, etc.)",
    "Otro"
]

# ============================================================
# NODOS (PRO173) — flujo corregido según indicaciones:
# - Post "Notificar trabajos" => rombo "¿Se requiere un informe...?"
# - "Realizar informe" => "Revisar informe..." => rombo "¿Informe completo?"
# - "¿El problema fue solucionado?" queda al FINAL, después de "Fin de mantenimiento"
# ============================================================
NODOS = {'S0_inicio': {'type': 'decision',
               'titulo': '¿Cómo iniciar el PRO173?',
               'rol': 'HMI (selección de inicio)',
               'descripcion': 'Seleccione el punto de entrada operativo según el contexto real (en terreno / '
                              'SAP).',
               'pregunta': 'Seleccione una opción para iniciar:',
               'opciones': [{'label': 'Inicio 1: Liberación de aviso A0/A1/A2',
                             'next': 'I1_Liberacion_aviso_A0A1A2'},
                            {'label': 'Inicio 2: Liberación de aviso A3', 'next': 'I2_Liberacion_aviso_A3'}],
               'ayuda': 'Selector HMI: no modifica el flujo del PRO, solo define desde dónde comenzar.'},
 'I1_Liberacion_aviso_A0A1A2': {'type': 'task',
                                'titulo': 'Liberación de aviso A0/A1/A2',
                                'rol': 'Supervisión de Mantenimiento',
                                'descripcion': 'Entrada del flujo por liberación de aviso A0/A1/A2 (según '
                                               'diagrama del PRO173).',
                                'acciones': ['Confirmar que el aviso está liberado y habilita la gestión del '
                                             'correctivo.'],
                                'checklist': ['Aviso A0/A1/A2 liberado confirmado',
                                              'Corresponde iniciar correctivo'],
                                'validacion': '¿El aviso A0/A1/A2 está liberado y corresponde iniciar la '
                                              'gestión del correctivo?',
                                'next': 'T1_Crear_OT_OT01'},
 'I2_Liberacion_aviso_A3': {'type': 'task',
                            'titulo': 'Liberación de aviso A3',
                            'rol': 'Supervisión de Mantenimiento',
                            'descripcion': 'Entrada del flujo por liberación de aviso A3 (según diagrama del '
                                           'PRO173).',
                            'acciones': ['Confirmar que el aviso A3 está liberado y habilita la gestión del '
                                         'correctivo.'],
                            'checklist': ['Aviso A3 liberado confirmado', 'Corresponde iniciar correctivo'],
                            'validacion': '¿El aviso A3 está liberado y corresponde iniciar la gestión del '
                                          'correctivo?',
                            'next': 'T1_Crear_OT_OT01'},
 'T1_Crear_OT_OT01': {'type': 'task',
                      'titulo': 'Crear OT (OT01)',
                      'rol': 'Especialización de Mantenimiento',
                      'descripcion': 'Crear Orden de Trabajo manual.\n'
                                     '- Crear OT01*\n'
                                     '- Definir una prioridad para su ejecución\n'
                                     '- Definir una denominación\n'
                                     '- Asignar el sistema/proceso (UT-Proceso)\n'
                                     '- Completar campos obligatorios.\n'
                                     '(*) Clase de actividad MCF para órdenes creadas a partir de avisos A0, '
                                     'A1 y A2\n'
                                     '(*) Clase de actividad MPC para órdenes creadas a partir de avisos A3\n'
                                     'En caso de que operaciones requiera una prueba funcional para evaluar '
                                     'la efectividad, se debe incorporar una operación adicional de prueba '
                                     'de funcionalidad del equipo.\n'
                                     'Transacción: IW31',
                      'acciones': ['Crear OT01 manual en IW31.',
                                   'Definir prioridad, denominación y UT-Proceso.',
                                   'Completar campos obligatorios.',
                                   'Si aplica, incorporar operación adicional de prueba funcional.'],
                      'checklist': ['OT creada en IW31',
                                    'Prioridad definida',
                                    'Denominación definida',
                                    'UT-Proceso asignada',
                                    'Campos obligatorios completos',
                                    'Si aplica: operación de prueba funcional incorporada'],
                      'validacion': '¿La OT quedó creada en IW31 con prioridad, denominación, UT-Proceso y '
                                    'campos obligatorios completos (y prueba funcional si aplica)?',
                      'next': 'T2_Preparar_OT'},
 'T2_Preparar_OT': {'type': 'task',
                    'titulo': 'Preparar OT',
                    'rol': 'Especialización de mantenimiento',
                    'descripcion': 'Documentar el requerimiento de mantenimiento.\n'
                                   '- Definir las tares de mantenimiento (operaciones)\n'
                                   '- Definir duración de los trabajos y N° de personal requerido por '
                                   'tarea.\n'
                                   '- Definir el Ubicación Técnica por función.\n'
                                   '- Definir de materiales requeridos.\n'
                                   '- Incorporar Servicios como Operación (PM03)\n'
                                   'Transacción: IW32',
                    'acciones': ['En IW32, documentar requerimiento y definir operaciones.',
                                 'Definir duración y dotación por tarea.',
                                 'Definir Ubicación Técnica por función.',
                                 'Definir materiales requeridos.',
                                 'Incorporar servicios como operación (PM03) si aplica.'],
                    'checklist': ['Operaciones definidas en OT',
                                  'Duración definida',
                                  'Dotación definida',
                                  'Ubicación Técnica por función definida',
                                  'Si aplica: Materiales requeridos definidos',
                                  'Si aplica: servicios incorporados como operación (PM03)'],
                    'validacion': '¿La OT quedó preparada en IW32 con operaciones, duración/dotación, UT por '
                                  'función, materiales y servicios (PM03) si aplica?',
                    'next': 'D1_Gestionar_recursos',
                    'text_inputs': [{'key': 'numero_ot',
                                     'label': 'Ingrese número de OT',
                                     'placeholder': 'Ej: 4500123456',
                                     'required': True}]},
 'T3_Liberar_OT': {'type': 'task',
                   'titulo': 'Liberar OT',
                   'rol': 'Supervisión de Mantenimiento',
                   'descripcion': 'Una vez creada y Preparada la OT, pasa por una revisión y confirmación de '
                                  'los trabajos requeridos por parte de la supervisión de mantenimiento '
                                  'llamada "Liberar OT".\n'
                                  'Transacción: IW32\n'
                                  'Consideración: En caso de que se evidencie previo a la liberación de una '
                                  'OT que esta no corresponde para su ejecución, se debe asignar el estatus '
                                  'NEJE a nivel de cabecera de la OT.',
                   'acciones': ['Revisar y confirmar trabajos requeridos.',
                                'Liberar OT en IW32.',
                                'Si NO corresponde ejecutar, asignar estatus NEJE (cabecera) antes de '
                                'liberar.'],
                   'checklist': ['OT revisada por supervisión',
                                 'OT liberada en IW32 o estatus NEJE aplicado (si no corresponde ejecutar)'],
                   'validacion': '¿La OT fue revisada y liberada correctamente (o se aplicó estatus NEJE si '
                                 'no corresponde ejecutar)?',
                   'next': 'T4_Preparar_trabajos_SI'},
 'T3_Liberar_OT_NO': {'type': 'task',
                      'titulo': 'Liberar OT',
                      'rol': 'Supervisión de Mantenimiento',
                      'descripcion': 'Una vez creada y Preparada la OT, pasa por una revisión y confirmación '
                                     'de los trabajos requeridos por parte de la supervisión de '
                                     'mantenimiento llamada "Liberar OT".\n'
                                     'Transacción: IW32\n'
                                     'Consideración: En caso de que se evidencie previo a la liberación de '
                                     'una OT que esta no corresponde para su ejecución, se debe asignar el '
                                     'estatus NEJE a nivel de cabecera de la OT.',
                      'acciones': ['Revisar y confirmar trabajos requeridos.',
                                   'Liberar OT en IW32.',
                                   'Si NO corresponde ejecutar, asignar estatus NEJE (cabecera) antes de '
                                   'liberar.'],
                      'checklist': ['OT revisada por supervisión',
                                    'OT liberada en IW32 o estatus NEJE aplicado (si no corresponde '
                                    'ejecutar)'],
                      'validacion': '¿La OT fue revisada y liberada correctamente (o se aplicó estatus NEJE '
                                    'si no corresponde ejecutar)?',
                      'next': 'T4_Preparar_trabajos_NO'},
 'T4_Preparar_trabajos_SI': {'type': 'task',
                             'titulo': 'Preparar trabajos',
                             'rol': 'Especialización de Mantenimiento',
                             'descripcion': 'Asegurar que tengo todo lo necesario para ejecutar los trabajos '
                                            'requeridos.\n'
                                            '- Procedimientos\n'
                                            '- Licitación de servicios\n'
                                            '- Permiso de Trabajo\n'
                                            '- Revisión de informes anteriores\n'
                                            'Transacción: No aplica',
                             'acciones': ['Verificar procedimientos aplicables.',
                                          'Verificar licitación/servicios si aplica.',
                                          'Verificar Permiso de Trabajo (PT).',
                                          'Revisar informes anteriores si existen.'],
                             'checklist': ['Procedimientos disponibles',
                                           'Servicios/licitación verificados (si aplica)',
                                           'PT considerado (previo a ejecución)',
                                           'Informes anteriores revisados (si aplica)'],
                             'validacion': '¿Está todo lo necesario para ejecutar (procedimientos, servicios '
                                           'si aplica, PT y revisión de informes anteriores)?',
                             'next': 'T5_Gestion_de_recursos',
                             'text_inputs': [{'key': 'numero_pt',
                                              'label': 'Ingrese número de PT',
                                              'placeholder': 'Ej: PT-2026-0001',
                                              'required': True}]},
 'T4_Preparar_trabajos_NO': {'type': 'task',
                             'titulo': 'Preparar trabajos',
                             'rol': 'Especialización de Mantenimiento',
                             'descripcion': 'Asegurar que tengo todo lo necesario para ejecutar los trabajos '
                                            'requeridos.\n'
                                            '- Procedimientos\n'
                                            '- Licitación de servicios\n'
                                            '- Permiso de Trabajo\n'
                                            '- Revisión de informes anteriores\n'
                                            'Transacción: No aplica',
                             'acciones': ['Verificar procedimientos aplicables.',
                                          'Verificar licitación/servicios si aplica.',
                                          'Verificar Permiso de Trabajo (PT).',
                                          'Revisar informes anteriores si existen.'],
                             'checklist': ['Procedimientos disponibles',
                                           'Servicios/licitación verificados (si aplica)',
                                           'PT considerado (previo a ejecución)',
                                           'Informes anteriores revisados (si aplica)'],
                             'validacion': '¿Está todo lo necesario para ejecutar (procedimientos, servicios '
                                           'si aplica, PT y revisión de informes anteriores)?',
                             'next': 'T8_Programacion_de_trabajos',
                             'text_inputs': [{'key': 'numero_pt',
                                              'label': 'Ingrese número de PT',
                                              'placeholder': 'Ej: PT-2026-0001',
                                              'required': True}]},
 'D1_Gestionar_recursos': {'type': 'decision',
                           'titulo': '¿Se requiere gestionar recursos?',
                           'rol': 'Planificación / Programación',
                           'descripcion': 'Decisión del flujo PRO173 para determinar si se requiere gestión '
                                          'de recursos (materiales/servicios).',
                           'pregunta': '¿Se necesitan recursos?',
                           'opciones': [{'label': 'SÍ', 'next': 'D2_Modo_CECO_PEP'},
                                        {'label': 'NO', 'next': 'T3_Liberar_OT_NO'}]},
 'T5_Gestion_de_recursos': {'type': 'task',
                            'titulo': 'Gestión de recursos',
                            'rol': 'Abastecimiento y Compras',
                            'descripcion': 'Gestión de recursos (según diagrama del PRO173).',
                            'acciones': ['Gestionar recursos requeridos (materiales/servicios) para '
                                         'habilitar ejecución.'],
                            'checklist': ['Recursos gestionados/confirmados'],
                            'validacion': '¿Los recursos necesarios fueron gestionados/confirmados para la '
                                          'ejecución?',
                            'next': 'T6_Recepcion_materiales_servicios'},
 'T6_Recepcion_materiales_servicios': {'type': 'task',
                                       'titulo': 'Recepción de materiales/servicios',
                                       'rol': 'Abastecimiento y Compras',
                                       'descripcion': 'Recepción de materiales/servicios (según diagrama del '
                                                      'PRO173).',
                                       'acciones': ['Confirmar recepción y disponibilidad de '
                                                    'materiales/servicios para ejecución.'],
                                       'checklist': ['Materiales recepcionados (si aplica)',
                                                     'Servicios disponibles/confirmados (si aplica)'],
                                       'validacion': '¿Materiales/servicios están recepcionados y '
                                                     'disponibles para ejecutar?',
                                       'next': 'T8_Programacion_de_trabajos'},
 'T8_Programacion_de_trabajos': {'type': 'task',
                                 'titulo': 'Programación de trabajos',
                                 'rol': 'Planificación / Programación',
                                 'descripcion': 'Programación de trabajos (según diagrama del PRO173).',
                                 'acciones': ['Programar ventana/fecha de ejecución y coordinación '
                                              'operativa.'],
                                 'checklist': ['Trabajo programado', 'Coordinación comunicada'],
                                 'validacion': '¿El trabajo quedó programado y coordinado para su ejecución?',
                                 'next': 'T7_PT_en_vigencia'},
 'T7_PT_en_vigencia': {'type': 'task',
                       'titulo': 'PT en estado “En vigencia”',
                       'rol': 'Planificación / Programación',
                       'descripcion': 'Condición habilitante para ejecución: PT en estado “En vigencia” '
                                      '(según diagrama del PRO173).',
                       'acciones': ['Verificar que el Permiso de Trabajo (PT) se encuentra vigente antes de '
                                    'ejecutar.'],
                       'checklist': ['PT vigente confirmado'],
                       'validacion': '¿El PT está en estado “En vigencia” y cubre el alcance real del '
                                     'trabajo?',
                       'next': None},
 'T9_Retirar_materiales': {'type': 'task',
                           'titulo': 'Retirar materiales',
                           'rol': 'Mantenimiento',
                           'descripcion': 'El mantenedor procede al retiro de materiales de bodega/almacén y '
                                          'solicita la aprobación de servicios externos contratados en caso '
                                          'de que esto aplique.\n'
                                          'Transacción: No aplica',
                           'acciones': ['Retirar materiales desde bodega/almacén.',
                                        'Solicitar aprobación de servicios externos contratados (si '
                                        'aplica).'],
                           'checklist': ['Materiales retirados',
                                         'Si aplica: aprobación servicios externos solicitada/obtenida'],
                           'validacion': '¿Materiales retirados y (si aplica) servicios externos aprobados '
                                         'para ejecutar?',
                           'next': 'T10_Ejecutar_trabajos'},
 'T10_Ejecutar_trabajos': {'type': 'task',
                           'titulo': 'Ejecutar trabajos',
                           'rol': 'Mantenimiento',
                           'descripcion': 'El mantenedor procederá con la ejecución de las tareas descritas '
                                          'como operaciones en la Orden de Trabajo.\n'
                                          'Transacción: No aplica',
                           'acciones': ['Ejecutar las operaciones de la OT en terreno, bajo PT vigente y '
                                        'estándares aplicables.'],
                           'checklist': ['Operaciones ejecutadas en terreno',
                                         'Condición segura mantenida durante ejecución'],
                           'validacion': '¿Se ejecutaron las operaciones de la OT en terreno bajo '
                                         'condiciones seguras y PT vigente?',
                           'next': 'D2_Reemplazo_equipo'},
 'D2_Reemplazo_equipo': {'type': 'decision',
                         'titulo': '¿El mantenimiento implicó reemplazo de equipo?',
                         'rol': 'Mantenimiento',
                         'descripcion': 'Decisión del flujo PRO173 respecto a reemplazo de equipo.',
                         'pregunta': '¿El mantenimiento implicó reemplazo de equipo?',
                         'opciones': [{'label': 'SÍ', 'next': 'T11_Gestion_de_Renovacion'},
                                      {'label': 'NO', 'next': 'T12_Notificar_trabajos'}]},
 'T11_Gestion_de_Renovacion': {'type': 'task',
                               'titulo': 'Gestión de Renovación',
                               'rol': 'Mantenimiento',
                               'descripcion': 'Gestión de Renovación (según diagrama del PRO173).',
                               'acciones': ['Derivar/iniciar gestión de renovación asociada al reemplazo del '
                                            'equipo.'],
                               'checklist': ['Gestión de renovación iniciada/derivada'],
                               'validacion': '¿Se inició/derivó correctamente la Gestión de Renovación por '
                                             'reemplazo de equipo?',
                               'next': 'T12_Notificar_trabajos'},
 'T12_Notificar_trabajos': {'type': 'task',
                            'titulo': 'Notificar trabajos',
                            'rol': 'Mantenimiento',
                            'descripcion': 'Una vez concluidos los trabajos, los mantenedores que '
                                           'participaron del trabajo realizado, deben proceder a registrar '
                                           'sus tiempos por operación asignada y para el caso de no haber '
                                           'realizado alguna tarea se debe dejar por escrito el motivo de la '
                                           'desviación.\n'
                                           'Transacción: IW41 / IW48',
                            'acciones': ['Notificar trabajos en IW41 o IW48.',
                                         'Registrar tiempos por operación asignada.',
                                         'Si no se realizó alguna tarea, dejar por escrito el motivo de la '
                                         'desviación.'],
                            'checklist': ['Notificación registrada (IW41/IW48)',
                                          'Tiempos por operación registrados',
                                          'Si aplica: desviaciones justificadas por escrito'],
                            'validacion': '¿Se notificaron los trabajos con tiempos por operación y (si '
                                          'aplica) desviaciones justificadas por escrito?',
                            'next': 'D4_Informe_requerido'},
 'D4_Informe_requerido': {'type': 'decision',
                          'titulo': '¿Se requiere un informe de los trabajos realizados?',
                          'rol': 'Supervisión de Mantenimiento',
                          'descripcion': 'Decisión del supervisor de mantenimiento según PRO173.',
                          'pregunta': '¿Se requiere un informe de los trabajos realizados?',
                          'opciones': [{'label': 'SÍ', 'next': 'T13_Realizar_informe'},
                                       {'label': 'NO', 'next': 'T15_Revisar_OT'}]},
 'T13_Realizar_informe': {'type': 'task',
                          'titulo': 'Realizar informe de trabajos realizados',
                          'rol': 'Mantenimiento',
                          'descripcion': 'Es decisión del supervisor de mantenimiento definir si los '
                                         'trabajos realizados, requieren un informe. Algunas de las '
                                         'situaciones donde se puede dar son:\n'
                                         '▪ La especificidad de los trabajos amerita documentar y evidenciar '
                                         'lo realizado.\n'
                                         '▪ Actividades en las cuales se realizan mediciones que se deben '
                                         'documentar.\n'
                                         '▪ Trabajos con servicios externos, entre otros.\n'
                                         'Este informe debe ser realizado por el ejecutor de los trabajos y '
                                         'debe incluir los detalles de lo realizado.\n'
                                         'Transacción: No aplica',
                          'acciones': ['Realizar informe de trabajos realizados con detalle de lo ejecutado.',
                                       'Incluir mediciones/evidencia cuando corresponda.',
                                       'Incluir participación de terceros/servicios cuando aplique.'],
                          'checklist': ['Informe elaborado por ejecutor',
                                        'Detalle de lo realizado incluido',
                                        'Si aplica: mediciones/evidencia incorporadas',
                                        'Si aplica: participación de terceros documentada'],
                          'validacion': '¿El informe fue realizado por el ejecutor e incluye el detalle de '
                                        'lo realizado (y mediciones/evidencia si aplica)?',
                          'next': 'T14_Revisar_informe_y_cargar'},
 'T14_Revisar_informe_y_cargar': {'type': 'task',
                                  'titulo': 'Revisar informe de trabajos realizados y cargar en SAP',
                                  'rol': 'Supervisión de Mantenimiento',
                                  'descripcion': 'El supervisor de mantenimiento debe revisar el informe '
                                                 'realizado y evaluar si este se encuentra completo.\n'
                                                 'Luego debe almacenar el informe en el gestor documental de '
                                                 'la compañía (Sharepoint).\n'
                                                 'Transacción: No aplica',
                                  'acciones': ['Revisar informe realizado.',
                                               'Evaluar completitud del informe.',
                                               'Almacenar informe en Sharepoint (gestor documental).'],
                                  'checklist': ['Informe revisado por supervisor',
                                                'Informe almacenado en Sharepoint'],
                                  'validacion': '¿El supervisor revisó el informe y quedó almacenado en '
                                                'Sharepoint?',
                                  'next': 'D5_Informe_completo'},
 'D5_Informe_completo': {'type': 'decision',
                         'titulo': '¿Informe completo?',
                         'rol': 'Supervisión de Mantenimiento',
                         'descripcion': 'Rombo del flujo PRO173 para validar completitud del informe.',
                         'pregunta': '¿Informe completo?',
                         'opciones': [{'label': 'SÍ', 'next': 'T15_Revisar_OT'},
                                      {'label': 'NO', 'next': 'T13_Realizar_informe'}]},
 'T15_Revisar_OT': {'type': 'task',
                    'titulo': 'Revisar OT',
                    'rol': 'Supervisión de Mantenimiento',
                    'descripcion': 'El supervisor revisará desde la OT que los trabajos fueron ejecutados y '
                                   'notificados, conforme a los recursos y tiempos planificados.\n'
                                   'Transacción: IW32 / IW33',
                    'acciones': ['Revisar en IW32/IW33 consistencia entre ejecución/notificación y plan '
                                 '(recursos/tiempos).'],
                    'checklist': ['OT revisada en IW32/IW33',
                                  'Consistencia ejecución/notificación vs plan confirmada'],
                    'validacion': '¿La OT fue revisada en IW32/IW33 y es consistente con recursos/tiempos '
                                  'planificados?',
                    'next': 'T16_Cerrar_OT_CTEC'},
 'T16_Cerrar_OT_CTEC': {'type': 'task',
                        'titulo': 'Cerrar OT (CTEC)',
                        'rol': 'Supervisión de Mantenimiento',
                        'descripcion': 'Una vez concluidos los trabajos de mantenimiento y notificados en el '
                                       'sistema, el supervisor procede a cerrar técnicamente la OT.\n'
                                       'Transacción: IW32',
                        'acciones': ['Cerrar técnicamente la OT (CTEC) en IW32.'],
                        'checklist': ['OT cerrada técnicamente (CTEC)'],
                        'validacion': '¿La OT fue cerrada técnicamente (CTEC) en IW32?',
                        'next': 'D6_A_la_falla'},
 'D6_A_la_falla': {'type': 'decision',
                   'titulo': '¿El equipo está con estrategia de mantenimiento “a la falla”?',
                   'rol': 'Supervisión de Mantenimiento',
                   'descripcion': 'Rombo del flujo PRO173 para determinar si aplica análisis de efectividad.',
                   'pregunta': '¿El equipo está con estrategia de mantenimiento “a la falla”?',
                   'opciones': [{'label': 'SÍ', 'next': 'T19_Fin_de_mantenimiento'},
                                {'label': 'NO', 'next': 'T17_Analisis_efectividad'}]},
 'T17_Analisis_efectividad': {'type': 'task',
                              'titulo': 'Realizar análisis de efectividad de controles definidos',
                              'rol': 'Supervisión de Mantenimiento',
                              'descripcion': 'En caso de que el equipo en falla este cubierto por el '
                                             'mantenimiento preventivo, se realiza el análisis de '
                                             'efectividad para detectar potenciales oportunidades de mejora '
                                             'en el plan de mantenimiento.\n'
                                             'Transacción: No aplica',
                              'acciones': ['Realizar análisis de efectividad de controles definidos y '
                                           'levantar oportunidades de mejora (si aplica).'],
                              'checklist': ['Análisis de efectividad realizado',
                                            'Oportunidades de mejora identificadas (si aplica)'],
                              'validacion': '¿Se realizó el análisis de efectividad y se identificaron '
                                            'oportunidades de mejora cuando corresponde?',
                              'next': 'D7_Modificar_plan'},
 'D7_Modificar_plan': {'type': 'decision',
                       'titulo': '¿Se debe modificar el plan de mantenimiento establecido?',
                       'rol': 'Supervisión de Mantenimiento',
                       'descripcion': 'Rombo del flujo PRO173 que deriva a PRO181 si corresponde.',
                       'pregunta': '¿Se debe modificar el plan de mantenimiento establecido?',
                       'opciones': [{'label': 'SÍ', 'next': 'T18_PRO181'},
                                    {'label': 'NO', 'next': 'T19_Fin_de_mantenimiento'}]},
 'T18_PRO181': {'type': 'task',
                'titulo': 'PRO181 - Modificación del Plan de Mantenimiento',
                'rol': 'Supervisión de Mantenimiento',
                'descripcion': 'Derivación externa según flujo PRO173: ejecutar PRO181 para modificar el '
                               'Plan de Mantenimiento.',
                'acciones': ['Derivar y ejecutar PRO181 según corresponda.'],
                'checklist': ['Derivación a PRO181 realizada', 'Responsable definido para ejecutar PRO181'],
                'validacion': '¿Se derivó correctamente a PRO181 con responsable definido?',
                'next': 'T19_Fin_de_mantenimiento'},
 'T19_Fin_de_mantenimiento': {'type': 'task',
                              'titulo': 'Fin de mantenimiento',
                              'rol': 'HMI',
                              'descripcion': 'Se ejecutaron los pasos del mantenimiento correctivo según '
                                             'PRO173. Corresponde registrar el resultado final del '
                                             'correctivo.',
                              'acciones': ['Registrar cierre operacional del mantenimiento (según '
                                           'procedimiento local) y confirmar condición final.'],
                              'checklist': ['Cierre operacional registrado', 'Condición final confirmada'],
                              'validacion': '¿Se registró el cierre operacional y se confirmó la condición '
                                            'final del equipo/sistema?',
                              'next': 'D3_Problema_solucionado'},
 'D3_Problema_solucionado': {'type': 'decision',
                             'titulo': '¿El problema fue solucionado?',
                             'rol': 'Supervisión de Mantenimiento',
                             'descripcion': 'Confirmación final del resultado del mantenimiento correctivo. '
                                            "Si NO fue solucionado, el flujo retorna a 'Crear OT (OT01)' "
                                            'para re-ejecutar el correctivo.',
                             'pregunta': '¿El problema fue solucionado?',
                             'opciones': [{'label': 'SÍ', 'next': 'T_UPLOAD_ARCHIVOS'},
                                          {'label': 'NO', 'next': 'T1_Crear_OT_OT01'}]},
 'END_OK': {'type': 'end',
            'titulo': '🏁 Proceso finalizado',
            'rol': 'HMI',
            'descripcion': 'Se completó el PRO173 y el problema fue solucionado. Puede exportar el JSON '
                           'auditable si lo requiere.',
            'mensaje': 'Ya está en el final/cierre del procedimiento.',
            'estado_final': 'FINALIZADO'},
 'END_NO': {'type': 'end',
            'titulo': '🏁 Proceso finalizado',
            'rol': 'HMI',
            'descripcion': 'Se completó el PRO173 y el problema NO fue solucionado (resultado informado). '
                           'Puede exportar el JSON auditable si lo requiere.',
            'mensaje': 'Ya está en el final/cierre del procedimiento.',
            'estado_final': 'FINALIZADO'},
 'D2_Modo_CECO_PEP': {'type': 'decision',
                      'titulo': '¿Qué modo escoges?',
                      'rol': 'Supervisor de Mantenimiento',
                      'descripcion': 'Seleccione el modo de imputación según corresponda: CECO o PEP.',
                      'pregunta': '¿Qué modo escoges?',
                      'opciones': [{'label': 'CECO', 'next': 'T2B_Asignar_CECO'},
                                   {'label': 'PEP', 'next': 'T2A_Asignar_PEP'}],
                      'ayuda': "Seleccione solo una opción. Ambos caminos convergen en 'Liberar OT'."},
 'T2A_Asignar_PEP': {'type': 'task',
                     'titulo': 'Asignar PEP',
                     'rol': 'Supervisor de Mantenimiento',
                     'descripcion': "Asignar el código del elemento PEP en la OT (pestaña 'Datos Adic.').",
                     'acciones': ['Definir tipo de PEP aplicable y registrar el número de PEP en la OT.'],
                     'checklist': ['Selecciona un tipo de PEP (excluyente)'],
                     'radio': {'key': 'pep_tipo',
                               'question': '¿Cuál aplica?',
                               'options': ['Mantenimiento mayor', 'Caso base'],
                               'required': True},
                     'text_inputs': [{'key': 'numero_pep',
                                      'label': 'Ingrese número de PEP',
                                      'placeholder': 'Ej: PEP-12345',
                                      'required': True}],
                     'validacion': '¿PEP asignado correctamente en la OT?',
                     'next': 'T3_Liberar_OT'},
 'T2B_Asignar_CECO': {'type': 'task',
                      'titulo': 'Asignar CECO',
                      'rol': 'Supervisor de Mantenimiento',
                      'descripcion': 'Asignar el número de CECO en la OT. Aplica: Mantenimiento cotidiano.',
                      'acciones': ['Registrar el número de CECO correspondiente en la OT.'],
                      'checklist': ['Aplica: Mantenimiento cotidiano'],
                      'text_inputs': [{'key': 'numero_ceco',
                                       'label': 'Ingrese número de CECO',
                                       'placeholder': 'Ej: CECO-0001',
                                       'required': True}],
                      'validacion': '¿CECO asignado en la OT?',
                      'next': 'T3_Liberar_OT'},
 'T_UPLOAD_ARCHIVOS': {'type': 'task',
                       'titulo': 'Opcional: Subir archivos pertinentes',
                       'rol': 'Mantenimiento',
                       'descripcion': 'Opcional. Registre enlaces/rutas/nombres de los archivos pertinentes '
                                      '(fotos, informes, evidencias).',
                       'acciones': ['Adjuntar o registrar referencias a archivos pertinentes.'],
                       'checklist': [],
                       'text_inputs': [{'key': 'archivos_pertinentes',
                                        'label': 'Opcional: Sube los archivos pertinentes',
                                        'placeholder': 'Pegue links / rutas / nombres de archivos...',
                                        'required': False,
                                        'multiline': True}],
                       'validacion': 'Paso opcional.',
                       'next': 'END_OK'}}



# -------------------------
# HMI (estilo visual PRO134 aprobado)
# -------------------------
class PRO173HMI:
    def __init__(self):
        self.nodo_id = "S0_inicio"
        self.historial = []
        self.logs = []
        self.decisiones = []
        self.bloqueos = []

        self.run_id = str(uuid.uuid4())
        self.inputs = {}
        self.estado = EN_CURSO
        self.start_ts = _now_iso()
        self.end_ts = None

        self.output = widgets.Output(layout={"width":"100%"})

        self.btn_si = widgets.Button(description="SÍ", button_style="success", layout={"width":"48%","height":"44px"})
        self.btn_no = widgets.Button(description="NO", button_style="danger", layout={"width":"48%","height":"44px"})
        self.btn_volver = widgets.Button(description="⬅ Volver al paso anterior", layout={"width":"100%","height":"40px"})
        self.btn_exportar = widgets.Button(description="Exportar JSON (trazabilidad)", icon="download", layout={"width":"100%","height":"40px"})

        self.msg_box = widgets.HTML("")
        self.main_box = widgets.VBox([])

        self.is_blocked = False
        self.block_panel = widgets.VBox([])
        self.btn_rehacer = widgets.Button(description="🔄 Rehacer paso", button_style="info", layout={"width":"100%","height":"40px"})
        self.btn_rehacer.on_click(self._on_rehacer)

        self._decision_widget = None
        self._check_widgets = []

        self._wire()
        self._render()

    def _wire(self):
        self.btn_si.on_click(self._on_si)
        self.btn_no.on_click(self._on_no)
        self.btn_volver.on_click(self._on_volver)
        self.btn_exportar.on_click(self._on_exportar)

    def _log(self, tipo, data=None):
        self.logs.append({
            "ts": _now_iso(),
            "tipo": tipo,
            "estado": self.estado,
            "nodo": self.nodo_id,
            "data": data or {}
        })

    def _push_hist(self):
        self.historial.append(self.nodo_id)

    def _pop_hist(self):
        if self.historial:
            return self.historial.pop()
        return None

    def _set_msg(self, html):
        self.msg_box.value = html

    def _clear_msg(self):
        self.msg_box.value = ""

    def _render_header(self, n):
        badge = f"<span style='display:inline-block;padding:4px 10px;border-radius:999px;background:#eef2ff;border:1px solid #c7d2fe;font-size:12px;color:black;'><b>ROL:</b> {n.get('rol','')}</span>"
        return widgets.HTML(f"""
        <div style="padding:16px;border-radius:12px;background:#f8fafc;border:1px solid #e2e8f0;">
            <div style="font-size:12px;color:#0f172a;"><b>PRO173</b> – Mantenimiento Correctivo</div>
            <div style="margin-top:6px;font-size:22px;color:#0f172a;"><b>{n.get('titulo','')}</b></div>
            <div style="margin-top:8px;">{badge}</div>
            <div style="margin-top:10px;color:#0f172a;font-size:14px;line-height:1.35;white-space:pre-wrap;">{n.get('descripcion','')}</div>
        </div>
        """)

    def _render_task(self, n):
        acciones = "".join([f"<li style='margin:4px 0;color:#0f172a;'>{a}</li>" for a in n.get("acciones",[])])
        valid = n.get("validacion","")

        # Checklist
        self._check_widgets = [
            widgets.Checkbox(description=item, value=False, layout=widgets.Layout(width="100%"))
            for item in (n.get("checklist",[]) or [])
        ]

        # Inputs (text / textarea)
        self._input_widgets = {}
        inputs_box = widgets.VBox([])
        text_inputs = n.get("text_inputs", []) or []
        if text_inputs:
            rows = []
            for spec in text_inputs:
                key = spec.get("key")
                label = spec.get("label", key)
                placeholder = spec.get("placeholder","")
                required = bool(spec.get("required", False))
                multiline = bool(spec.get("multiline", False))

                existing = self.inputs.get(key, "")
                if multiline:
                    w = widgets.Textarea(
                        value=existing,
                        placeholder=placeholder,
                        layout=widgets.Layout(width="100%", height="90px")
                    )
                else:
                    w = widgets.Text(
                        value=existing,
                        placeholder=placeholder,
                        layout=widgets.Layout(width="100%")
                    )
                self._input_widgets[key] = (w, required, label)

                # Mejor legibilidad en tema oscuro
                if key in ('numero_ot','numero_pt','numero_pep','numero_ceco'):
                    # Forzar texto blanco en inputs numéricos (tema oscuro)
                    try:
                        w.add_class('input-white')
                    except Exception:
                        pass
                    try:
                        if hasattr(w, "_dom_classes"):
                            w._dom_classes.add("input-white")
                    except Exception:
                        pass


                req_tag = " <span style='color:#ef4444;'><b>(obligatorio)</b></span>" if required else " <span style='color:#64748b;'>(opcional)</span>"
                rows.append(widgets.VBox([
                    widgets.HTML(f"<div style='margin-top:10px;color:#0f172a;'><b>{label}</b>{req_tag}</div>"),
                    w
                ]))
            inputs_box = widgets.VBox([
                widgets.HTML('''
                <div style="margin-top:12px;padding:14px;border-radius:12px;border:1px solid #e2e8f0;background:#ffffff;">
                    <div style="font-size:13px;color:#0f172a;"><b>🧩 CAMPOS</b></div>
                </div>
                '''),
                widgets.VBox(rows)
            ])

        # Radio (selección excluyente) - opcional por nodo
        self._radio_widget = None
        self._radio_spec = None
        radio_box = widgets.VBox([])
        radio_spec = n.get("radio")
        if radio_spec:
            self._radio_spec = radio_spec
            opts = radio_spec.get("options", [])
            self._radio_widget = widgets.RadioButtons(
                options=opts,
                layout={"width":"100%"},
                style={"description_width":"initial"},
            )
            existing = self.inputs.get(radio_spec.get("key",""), None)
            if existing in opts:
                self._radio_widget.value = existing

            req = bool(radio_spec.get("required", False))
            req_tag = " <span style='color:#ef4444;'><b>(obligatorio)</b></span>" if req else " <span style='color:#64748b;'>(opcional)</span>"
            question = radio_spec.get("question", "Seleccione una opción:")
            radio_box = widgets.VBox([
                widgets.HTML(f'''
                <div style="margin-top:12px;padding:14px;border-radius:12px;border:1px solid #e2e8f0;background:#ffffff;">
                    <div style="font-size:13px;color:#0f172a;"><b>🔘 SELECCIÓN</b></div>
                    <div style="margin-top:8px;font-size:14px;color:#0f172a;"><b>{question}</b>{req_tag}</div>
                </div>
                '''),
                self._radio_widget
            ])

        accion_box = widgets.HTML(f'''
            <div style="margin-top:12px;padding:14px;border-radius:12px;border:1px solid #e2e8f0;background:#ffffff;">
                <div style="font-size:13px;color:#0f172a;"><b>⚙️ ACCIÓN A EJECUTAR (texto PRO173)</b></div>
                <ul style="margin-top:10px;padding-left:18px;color:#0f172a;">{acciones}</ul>
            </div>
        ''')

        checklist_box = widgets.VBox([])
        if self._check_widgets:
            checklist_box = widgets.VBox([
                widgets.HTML('''
                <div style="margin-top:12px;padding:14px;border-radius:12px;border:1px solid #e2e8f0;background:#ffffff;">
                    <div style="font-size:13px;color:#0f172a;"><b>🧾 CHECKLIST</b></div>
                    <div style="margin-top:8px;font-size:12px;color:#0f172a;opacity:0.9;">
                        Ítems marcados como <b>“Si aplica:”</b> no son obligatorios para avanzar.
                    </div>
                </div>
                '''),
                widgets.VBox(self._check_widgets)
            ])

        valid_box = widgets.HTML(f'''
            <div style="margin-top:12px;padding:14px;border-radius:12px;border:2px solid #0ea5e9;background:#ffffff;">
                <div style="font-size:13px;color:#0f172a;"><b>✅ ¡VALIDACIÓN!</b></div>
                <div style="margin-top:8px;font-size:16px;color:#0f172a;"><b>{valid}</b></div>
                <div style="margin-top:6px;font-size:12px;color:#0f172a;">Confirma con <b>SÍ</b> para avanzar. Si respondes <b>NO</b>, el paso queda bloqueado.</div>
            </div>
        ''')

        return widgets.VBox([accion_box, checklist_box, inputs_box, radio_box, valid_box])

    def _render_decision(self, n):
        opts = n.get("opciones",[])
        radios = widgets.RadioButtons(
            options=[(o["label"], o["next"]) for o in opts],
            layout={"width":"100%"},
            style={"description_width":"initial"},
        )
        help_txt = n.get("ayuda","")
        help_html = f"<div style='margin-top:10px;font-size:12px;color:#0f172a;opacity:0.9;'><b>Nota:</b> {help_txt}</div>" if help_txt else ""
        return widgets.VBox([
            widgets.HTML(f"""
            <div style="margin-top:12px;padding:14px;border-radius:12px;border:1px solid #e2e8f0;background:#ffffff;">
                <div style="font-size:13px;color:#0f172a;"><b>🔶 DECISIÓN (rombo)</b></div>
                <div style="margin-top:8px;font-size:16px;color:#0f172a;"><b>{n.get('pregunta','')}</b></div>
                {help_html}
            </div>
            """),
            radios
        ]), radios

    def _render_block_panel(self):
        if not self.is_blocked:
            self.block_panel.children = []
            return

        title = widgets.HTML("""
        <div style="margin-top:12px;padding:14px;border-radius:12px;border:2px solid #ef4444;background:#fff1f2;">
            <div style="font-size:14px;color:#0f172a;"><b>⛔ BLOQUEADO</b> — Seleccione motivo(s) y registre detalle.</div>
            <div style="margin-top:8px;font-size:12px;color:#0f172a;">No puede avanzar hasta rehacer el paso.</div>
        </div>
        """)

        self.sel_motivos = widgets.SelectMultiple(options=MOTIVOS_BLOQUEO_PRO173, rows=8, layout={"width":"100%"})
        self.txt_detalle = widgets.Textarea(
            placeholder="Detalle del bloqueo (obligatorio si selecciona 'Otro').",
            layout=widgets.Layout(width="100%", height="80px")
        )

        self.block_panel.children = [
            title,
            widgets.HTML("<b>Motivo(s) de bloqueo:</b> (selección múltiple)"),
            self.sel_motivos,
            widgets.HTML("<b>Detalle:</b>"),
            self.txt_detalle,
            self.btn_rehacer
        ]

    def _render_footer(self):
        self.btn_volver.disabled = (len(self.historial) == 0)
        return widgets.VBox([
            widgets.HBox([self.btn_si, self.btn_no], layout={"justify_content":"space-between","margin":"10px 0"}),
            self.btn_volver,
            widgets.HTML("<div style='height:10px;'></div>"),
            self.btn_exportar,
            widgets.HTML("<div style='height:10px;'></div>"),
            self.block_panel,
            self.msg_box,
        ])

    def _render(self):
        with self.output:
            self.output.clear_output()
            self._clear_msg()

            n = NODOS[self.nodo_id]
            header = self._render_header(n)

            if n["type"] == "task":
                body = self._render_task(n)
                self._decision_widget = None
            elif n["type"] == "decision":
                if n.get("auto_route"):
                    # Nodo técnico: se resuelve solo
                    necesita = bool(self.inputs.get("necesita_recursos", False))
                    ruta_txt = "Recursos: SÍ" if necesita else "Recursos: NO"
                    body = widgets.HTML(f'''
                    <div style="margin-top:12px;padding:14px;border-radius:12px;border:1px solid #e2e8f0;background:#ffffff;">
                        <div style="font-size:13px;color:#0f172a;"><b>🔁 Ruta automática</b></div>
                        <div style="margin-top:8px;font-size:14px;color:#0f172a;">{ruta_txt}. Presione <b>SÍ</b> para continuar.</div>
                    </div>
                    ''')
                    self._decision_widget = None
                    self._check_widgets = []
                else:
                    body, radios = self._render_decision(n)
                    self._decision_widget = radios
                    self._check_widgets = []
            elif n["type"] == "end":
                self._decision_widget = None
                self._check_widgets = []
                # Tabla resumen de inputs
                ot = self.inputs.get("numero_ot","")
                modo = self.inputs.get("modo","")
                pep = self.inputs.get("numero_pep","") if modo == "PEP" else ""
                ceco = self.inputs.get("numero_ceco","") if modo == "CECO" else ""
                pt = self.inputs.get("numero_pt","")
                # Tabla decisiones
                rows_dec = ""
                for d in self.decisiones:
                    rows_dec += f"<tr><td style='border:1px solid #e2e8f0;padding:6px;'>{d.get('ts','')}</td><td style='border:1px solid #e2e8f0;padding:6px;'>{d.get('nodo','')}</td><td style='border:1px solid #e2e8f0;padding:6px;'>{d.get('seleccion','')}</td></tr>"
                if not rows_dec:
                    rows_dec = "<tr><td colspan='3' style='border:1px solid #e2e8f0;padding:6px;color:#64748b;'>Sin decisiones registradas.</td></tr>"

                body = widgets.HTML(f'''
                <div style="margin-top:12px;padding:18px;border-radius:12px;border:2px solid #22c55e;background:#f0fdf4;">
                    <div style="font-size:20px;color:#0f172a;"><b>🏁 FIN</b></div>
                    <div style="margin-top:10px;font-size:15px;color:#0f172a;">{n.get('mensaje','Proceso finalizado.')}</div>

                    <div style="margin-top:14px;padding:12px;border-radius:12px;border:1px solid #e2e8f0;background:#ffffff;">
                        <div style="font-size:13px;color:#0f172a;"><b>📌 Resumen</b></div>
                        <table style="margin-top:10px;border-collapse:collapse;width:100%;font-size:13px;color:#0f172a;">
                            <tr><th style='text-align:left;border:1px solid #e2e8f0;padding:6px;'>Campo</th><th style='text-align:left;border:1px solid #e2e8f0;padding:6px;'>Valor</th></tr>
                            <tr><td style='border:1px solid #e2e8f0;padding:6px;'>N° OT</td><td style='border:1px solid #e2e8f0;padding:6px;'>{ot}</td></tr>
                            <tr><td style='border:1px solid #e2e8f0;padding:6px;'>Modo</td><td style='border:1px solid #e2e8f0;padding:6px;'>{modo}</td></tr>
                            <tr><td style='border:1px solid #e2e8f0;padding:6px;'>N° PEP</td><td style='border:1px solid #e2e8f0;padding:6px;'>{pep}</td></tr>
                            <tr><td style='border:1px solid #e2e8f0;padding:6px;'>N° CECO</td><td style='border:1px solid #e2e8f0;padding:6px;'>{ceco}</td></tr>
                            <tr><td style='border:1px solid #e2e8f0;padding:6px;'>N° PT</td><td style='border:1px solid #e2e8f0;padding:6px;'>{pt}</td></tr>
                        </table>
                    </div>

                    <div style="margin-top:14px;padding:12px;border-radius:12px;border:1px solid #e2e8f0;background:#ffffff;">
                        <div style="font-size:13px;color:#0f172a;"><b>🧭 Decisiones registradas</b></div>
                        <table style="margin-top:10px;border-collapse:collapse;width:100%;font-size:12px;color:#0f172a;">
                            <tr><th style='text-align:left;border:1px solid #e2e8f0;padding:6px;'>TS</th><th style='text-align:left;border:1px solid #e2e8f0;padding:6px;'>Nodo</th><th style='text-align:left;border:1px solid #e2e8f0;padding:6px;'>Selección</th></tr>
                            {rows_dec}
                        </table>
                    </div>
                </div>
                ''')
            else:
                body = widgets.HTML("<div>Tipo de nodo no soportado.</div>")
                self._decision_widget = None
                self._check_widgets = []

            self._render_block_panel()
            footer = self._render_footer()
            self.main_box.children = [header, body, footer]
            display(self.main_box)

    def _check_ready_to_advance(self):
        n = NODOS[self.nodo_id]

        if self.is_blocked:
            return False, "Paso bloqueado. Registre motivo(s) y use 'Rehacer paso'."

        if n["type"] == "end":
            return True, ""

        if n["type"] == "decision":
            # Decisiones automáticas: se resuelven solas (sin interacción)
            if n.get("auto_route"):
                return True, ""
            if self._decision_widget is None or self._decision_widget.value is None:
                return False, "Debe seleccionar una opción para avanzar."
            return True, ""

        if n["type"] == "task":
            # Checklist: 'Si aplica:' no es obligatorio
            if self._check_widgets:
                for cb in self._check_widgets:
                    txt = (cb.description or "").strip()
                    if "si aplica" in txt.lower():
                        continue
                    if not cb.value:
                        return False, "Debe completar el checklist antes de avanzar."
            # Text inputs requeridos
            if getattr(self, "_input_widgets", None):
                for key, (w, required, label) in self._input_widgets.items():
                    if required and (w.value is None or str(w.value).strip() == ""):
                        return False, f"Debe completar el campo obligatorio: {label}."
            # Radio requerido
            if getattr(self, "_radio_spec", None) and self._radio_spec:
                if self._radio_spec.get("required"):
                    if self._radio_widget is None or self._radio_widget.value is None:
                        return False, "Debe seleccionar una opción para avanzar."
            return True, ""

        return True, ""

    def _advance_to(self, next_id):
        if next_id not in NODOS:
            self._set_msg(f"""
            <div style='margin-top:10px;padding:12px;border-radius:10px;background:#fff7ed;border:1px solid #fdba74;color:#0f172a;'>
                <b>⚠ Error de flujo:</b> el nodo destino no existe: <code>{next_id}</code>
            </div>
            """)
            self._log("ERROR_FLUJO", {"missing_next": next_id})
            return
        self.nodo_id = next_id
        self._render()

    def _on_si(self, _):
        ok, msg = self._check_ready_to_advance()
        if not ok:
            self._set_msg(f'''
            <div style='margin-top:10px;padding:12px;border-radius:10px;background:#fff1f2;border:1px solid #fecdd3;color:#0f172a;'>
                <b>⚠ {msg}</b>
            </div>
            ''')
            self._log("VALIDACION_FALLA", {"mensaje": msg})
            return

        n = NODOS[self.nodo_id]

        if n["type"] == "end":
            self.estado = FINALIZADO
            self.end_ts = _now_iso()
            self._log("FINALIZA")
            self._set_msg('''
            <div style='margin-top:10px;padding:12px;border-radius:10px;background:#f1f5f9;border:1px solid #cbd5e1;color:#0f172a;'>
                <b>🏁 Ya está en el final/cierre.</b>
            </div>
            ''')
            return

        # Persistir inputs (tasks)
        if n["type"] == "task":
            if getattr(self, "_input_widgets", None):
                for key, (w, _req, _label) in self._input_widgets.items():
                    self.inputs[key] = w.value
            if getattr(self, "_radio_spec", None) and self._radio_spec and self._radio_widget is not None:
                rkey = self._radio_spec.get("key")
                if rkey:
                    self.inputs[rkey] = self._radio_widget.value

        self._push_hist()

        if n["type"] == "task":
            # Enrutamiento sin nodo extra (PT -> según '¿Se necesitan recursos?')
            next_id = n.get("next")
            if self.nodo_id == "T7_PT_en_vigencia":
                necesita = bool(self.inputs.get("necesita_recursos", False))
                next_id = "T9_Retirar_materiales" if necesita else "T10_Ejecutar_trabajos"
            self._log("AVANZA", {"next": next_id})
            self._advance_to(next_id)
            return

        if n["type"] == "decision":
            # Decisión automática (sin interacción)
            if n.get("auto_route"):
                necesita = bool(self.inputs.get("necesita_recursos", False))
                # Convención: opción 0 = SÍ, opción 1 = NO
                chosen = n.get("opciones",[{}])[0]["next"] if necesita else n.get("opciones",[{},{}])[1]["next"]
                chosen_label = n.get("opciones",[{}])[0].get("label") if necesita else n.get("opciones",[{},{}])[1].get("label")
                self.decisiones.append({
                    "ts": _now_iso(),
                    "nodo": self.nodo_id,
                    "titulo": n.get("titulo",""),
                    "seleccion": chosen_label,
                    "next": chosen
                })
                self._log("DECISION", {"seleccion": chosen_label, "next": chosen, "auto": True})
                self._advance_to(chosen)
                return

            chosen_next = self._decision_widget.value
            chosen_label = next((o["label"] for o in n.get("opciones",[]) if o["next"] == chosen_next), None)

            # Guardar decisiones clave en inputs
            if self.nodo_id == "D1_Gestionar_recursos":
                self.inputs["necesita_recursos"] = (chosen_label == "SÍ")
            if self.nodo_id == "D2_Modo_CECO_PEP":
                self.inputs["modo"] = chosen_label

            self.decisiones.append({
                "ts": _now_iso(),
                "nodo": self.nodo_id,
                "titulo": n.get("titulo",""),
                "seleccion": chosen_label,
                "next": chosen_next
            })
            self._log("DECISION", {"seleccion": chosen_label, "next": chosen_next})
            self._advance_to(chosen_next)
            return

    def _on_no(self, _):
        if self.is_blocked:
            return
        self.is_blocked = True
        self.estado = BLOQUEADO
        self.block_ts_inicio = _now_iso()
        self._log("BLOQUEADO_INICIO")
        self._render()

    def _on_rehacer(self, _):
        motivos = list(self.sel_motivos.value) if hasattr(self, "sel_motivos") else []
        detalle = (self.txt_detalle.value or "").strip() if hasattr(self, "txt_detalle") else ""

        if not motivos:
            self._set_msg("""
            <div style='margin-top:10px;padding:12px;border-radius:10px;background:#fff1f2;border:1px solid #fecdd3;color:#0f172a;'>
                <b>⚠ Debe seleccionar al menos un motivo.</b>
            </div>
            """)
            self._log("BLOQUEADO_VALIDACION_FALLA", {"mensaje": "sin_motivo"})
            return

        if "Otro" in motivos and not detalle:
            self._set_msg("""
            <div style='margin-top:10px;padding:12px;border-radius:10px;background:#fff1f2;border:1px solid #fecdd3;color:#0f172a;'>
                <b>⚠ Debe ingresar detalle si selecciona 'Otro'.</b>
            </div>
            """)
            self._log("BLOQUEADO_VALIDACION_FALLA", {"mensaje": "otro_sin_detalle"})
            return

        bloqueo = {
            "ts_inicio": getattr(self, "block_ts_inicio", None),
            "ts_fin": _now_iso(),
            "nodo": self.nodo_id,
            "titulo": NODOS[self.nodo_id].get("titulo",""),
            "motivos": motivos,
            "detalle": detalle
        }
        self.bloqueos.append(bloqueo)
        self._log("BLOQUEADO_FIN", bloqueo)

        self.is_blocked = False
        self.estado = EN_CURSO
        self._log("REHACER_PASO")
        self._render()

    def _on_volver(self, _):
        if self.is_blocked:
            self._set_msg("""
            <div style='margin-top:10px;padding:12px;border-radius:10px;background:#fff1f2;border:1px solid #fecdd3;color:#0f172a;'>
                <b>⚠ No puede volver mientras el paso está bloqueado. Use 'Rehacer paso'.</b>
            </div>
            """)
            return

        prev_id = self._pop_hist()
        if prev_id is not None:
            self._log("VOLVER", {"to": prev_id})
            self._advance_to(prev_id)

    def _on_exportar(self, _):
        payload = {
            "proceso": "PRO173 – Mantenimiento Correctivo",
            "run_id": self.run_id,
            "estado": self.estado,
            "start_ts": self.start_ts,
            "end_ts": self.end_ts,
            "current_node": self.nodo_id,
            "history_stack": list(self.historial),
            "decisiones": list(self.decisiones),
            "bloqueos": list(self.bloqueos),
            "inputs": dict(self.inputs),
            "logs": list(self.logs),
            "export_ts": _now_iso(),
        }
        pretty = json.dumps(payload, ensure_ascii=False, indent=2)
        self._set_msg(f"""
        <div style='margin-top:10px;padding:12px;border-radius:10px;background:#f1f5f9;border:1px solid #cbd5e1;color:#0f172a;'>
            <b>📦 Export JSON (trazabilidad)</b>
            <pre style='white-space:pre-wrap;margin-top:10px;color:#0f172a;'>{pretty}</pre>
        </div>
        """)

    def iniciar(self):
        display(widgets.HTML("""<style>.input-white input, .input-white textarea, .input-white .widget-input, .input-white .widget-text input, .input-white .widget-textarea textarea {color:#ffffff !important; -webkit-text-fill-color:#ffffff !important;}
.input-white input::placeholder, .input-white textarea::placeholder {color:#cbd5e1 !important; -webkit-text-fill-color:#cbd5e1 !important;}
</style>"""))
        display(self.output)

hmi = PRO173HMI()
hmi.iniciar()

HTML(value='<style>.input-white input, .input-white textarea, .input-white .widget-input, .input-white .widget…

Output(layout=Layout(width='100%'))